# 01 — Data Audit

This notebook checks whether the current processed queue-time dataset is usable for analysis.

The goal is not to generate final insights yet. The goal is to understand coverage, missing values, attraction representation and whether we have enough snapshots to support stronger EDA and modeling.


In [2]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from parkflow.data.data_quality import (
    add_audit_time_columns,
    build_coverage_summary,
    hourly_coverage_report,
    load_best_available_dataset,
    missingness_report,
    ride_coverage_report,
)


## 1. Load the best available processed dataset


In [3]:
df, path = load_best_available_dataset()
path, df.shape


(WindowsPath('D:/Documents/GitHub/BCWflow/data/processed/modeling_dataset.csv'),
 (41, 29))

In [4]:
df = add_audit_time_columns(df)
df.head()


,park_id,land_id,land_name,ride_id,ride_name,is_open,wait_time,last_updated_utc,ingested_at_utc,source,...,relative_humidity_2m,precipitation,wind_speed_10m,weather_code,latitude,longitude,timezone,ingested_at_local,audit_date_local,audit_hour_local
0,319,NaN,NaN,11329,Autopista (bate-bate),True,10,2026-05-21 20:00:05+00:00,2026-05-21 20:01:51.104747+00:00,queue-times.com,...,68,0.0,18.6,0,-26.818981,-48.63788,America/Sao_Paulo,2026-05-21 17:01:51.104747-03:00,2026-05-21,17
1,319,NaN,NaN,11330,Baby Elefante,True,10,2026-05-21 20:00:05+00:00,2026-05-21 20:01:51.104747+00:00,queue-times.com,...,68,0.0,18.6,0,-26.818981,-48.63788,America/Sao_Paulo,2026-05-21 17:01:51.104747-03:00,2026-05-21,17
2,319,NaN,NaN,11340,Barco Pirata,True,10,2026-05-21 20:00:05+00:00,2026-05-21 20:01:51.104747+00:00,queue-times.com,...,68,0.0,18.6,0,-26.818981,-48.63788,America/Sao_Paulo,2026-05-21 17:01:51.104747-03:00,2026-05-21,17
3,319,NaN,NaN,13872,Betinho Carrero 2D,True,5,2026-05-21 20:00:05+00:00,2026-05-21 20:01:51.104747+00:00,queue-times.com,...,68,0.0,18.6,0,-26.818981,-48.63788,America/Sao_Paulo,2026-05-21 17:01:51.104747-03:00,2026-05-21,17
4,319,NaN,NaN,11459,Big Drop,True,15,2026-05-21 20:00:05+00:00,2026-05-21 20:01:51.104747+00:00,queue-times.com,...,68,0.0,18.6,0,-26.818981,-48.63788,America/Sao_Paulo,2026-05-21 17:01:51.104747-03:00,2026-05-21,17


## 2. High-level coverage summary


In [5]:
summary = build_coverage_summary(df)
pd.DataFrame(summary.items(), columns=['metric', 'value'])


,metric,value
0,rows,41
1,snapshots,1
2,attractions,41
3,first_snapshot_local,2026-05-21 17:01
4,last_snapshot_local,2026-05-21 17:01
5,days_covered,1
6,average_wait_min,7.317073
7,p90_wait_min,30.0
8,open_rate,1.0
9,positive_wait_rate,0.390244


## 3. Coverage by attraction

This table helps us identify whether some attractions are underrepresented, always closed or producing unusual wait-time values.


In [6]:
ride_report = ride_coverage_report(df)
ride_report.head(30)


,ride_name,rows,avg_wait_min,median_wait_min,p90_wait_min,max_wait_min,snapshots,open_rate,days_seen
0,Tigor Mountain,1,45.0,45.0,45.0,45,1,1.0,1
1,Roda-Gigante,1,40.0,40.0,40.0,40,1,1.0,1
2,TURBO DRIVE,1,40.0,40.0,40.0,40,1,1.0,1
3,Raskapuska,1,35.0,35.0,35.0,35,1,1.0,1
4,Ferrovia DinoMagic,1,30.0,30.0,30.0,30,1,1.0,1
5,Big Drop,1,15.0,15.0,15.0,15,1,1.0,1
6,Madagascar Crazy River Adventure!,1,15.0,15.0,15.0,15,1,1.0,1
7,SPIN BLAST,1,15.0,15.0,15.0,15,1,1.0,1
8,Autopista (bate-bate),1,10.0,10.0,10.0,10,1,1.0,1
9,Baby Elefante,1,10.0,10.0,10.0,10,1,1.0,1


## 4. Coverage by date and hour


In [7]:
hourly = hourly_coverage_report(df)
hourly.head(30)


,audit_date_local,audit_hour_local,rows,snapshots,attractions,avg_wait_min
0,2026-05-21,17,41,1,41,7.317073


In [8]:
if not hourly.empty and 'snapshots' in hourly.columns:
    coverage_pivot = hourly.pivot_table(
        index='audit_date_local',
        columns='audit_hour_local',
        values='snapshots',
        aggfunc='sum',
        fill_value=0,
    )
    display(coverage_pivot)


audit_hour_local,17
audit_date_local,
2026-05-21,1


## 5. Missingness report


In [9]:
missingness_report(df)


,column,missing_count,missing_pct,dtype
0,land_id,41,1.0,float64
1,land_name,41,1.0,float64
2,park_id,0,0.0,int64
3,ride_id,0,0.0,int64
4,ride_name,0,0.0,str
5,is_open,0,0.0,bool
6,wait_time,0,0.0,int64
7,last_updated_utc,0,0.0,"datetime64[us, UTC]"
8,ingested_at_utc,0,0.0,"datetime64[us, UTC]"
9,source,0,0.0,str


## 6. Initial decision checklist

Use this checklist after each collection cycle:

- Do we have more than one snapshot?
- Do we have multiple hours covered?
- Do we have more than one day covered?
- Are the timestamps aligned with local park time?
- Are most attractions represented consistently?
- Are wait times mostly zero because the park was closed, or because data is sparse?
- Is there enough coverage to make EDA claims without overinterpreting the data?
